In [17]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

load dataset


In [18]:
from google.collab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:



data = pd.read_csv('/content/drive/MyDrive/IMDB Dataset.csv')

In [20]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [21]:
data.shape

(50000, 2)

In [22]:
data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


One hot Encoding

In [23]:
data.replace({"sentiment" : {"positive" : 1, "negative" : 0}}, inplace=True)

Data Preprocessing

In [24]:
!pip install tensorflow

In [25]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [26]:
train_data , test_data = train_test_split(data, test_size=0.2, random_state=42)

In [27]:
train_data.shape

(40000, 2)

In [28]:
test_data.shape

(10000, 2)

In [29]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(test_data["review"])

In [30]:
x_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
x_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

In [31]:
x_train

array([[ 776, 2301,    1, ...,  210,  374, 4301],
       [  14,    3, 1495, ...,   89,  104,    9],
       [   0,    0,    0, ...,    2,  734,   65],
       ...,
       [   0,    0,    0, ..., 1491,    2,  603],
       [   0,    0,    0, ...,  235,  104,  124],
       [   0,    0,    0, ...,   68,   71, 2087]], dtype=int32)

In [32]:
x_test

array([[   0,    0,    0, ..., 1050,  746,  162],
       [  12,  159,   58, ...,  397,    7,    7],
       [   0,    0,    0, ...,   51, 1081,   99],
       ...,
       [   0,    0,    0, ...,  124,  198, 3295],
       [   0,    0,    0, ..., 1080,    1, 2473],
       [   0,    0,    0, ...,    1,  341,   28]], dtype=int32)

In [45]:
y_train = train_data["sentiment"]
y_test =  test_data["sentiment"]

In [34]:
y_train

,sentiment
39087,0
30893,0
45278,1
16398,0
13653,0
...,...
11284,1
44732,1
38158,0
860,1


In [35]:
y_test

,sentiment
39087,0
30893,0
45278,1
16398,0
13653,0
...,...
11284,1
44732,1
38158,0
860,1


Model Building

In [36]:
model = Sequential()
model.add(Embedding(input_dim = 5000,output_dim =  128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

In [37]:
model.summary

<bound method Model.summary of <Sequential name=sequential, built=False>>

In [39]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [40]:
model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split= 0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 216s 414ms/step - accuracy: 0.7086 - loss: 0.5468 - val_accuracy: 0.8420 - val_loss: 0.3709
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 205s 411ms/step - accuracy: 0.8562 - loss: 0.3521 - val_accuracy: 0.7269 - val_loss: 0.5333
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 206s 413ms/step - accuracy: 0.8207 - loss: 0.3993 - val_accuracy: 0.8737 - val_loss: 0.3146
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 206s 411ms/step - accuracy: 0.8887 - loss: 0.2728 - val_accuracy: 0.8788 - val_loss: 0.2957
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 207s 414ms/step - accuracy: 0.9089 - loss: 0.2273 - val_accuracy: 0.8824 - val_loss: 0.2919


In [46]:
loss,accuracy = model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 42s 132ms/step - accuracy: 0.8876 - loss: 0.2777


In [47]:
print(loss)
print(accuracy)

0.27654948830604553
0.8870999813079834


Building Predicting System


In [48]:
def Predictive_system(review):
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence, maxlen=200)
  prediction = model.predict(padded_sequence)
  sentiment = "positive" if prediction > 0.5 else "negative"
  return sentiment

In [49]:
Predictive_system("this movie shit and amazing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 421ms/step


'positive'

In [50]:
Predictive_system("this movie is not worth it")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step


'negative'

In [51]:
Predictive_system("movie is suck")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step


'negative'

In [52]:
Predictive_system("this movie shit and stunning")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step


'positive'

saving model

In [57]:
model.save("model.h5")

In [55]:
import joblib
joblib.dump(tokenizer, "tokenizer.pkl")

['tokenizer.pkl']